# Bayes' theorem

Bayes' theorem is the rule for changing your mind when you see evidence. It is also the basis of the naive Bayes classifier, and it explains regularisation in notebook 03.

Order: intuition, maths, idea, from scratch, library, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

## 1. Intuition

A test for a rare disease is 90% accurate. You test positive. How worried should you be?

Most people say 90%. The right answer depends on how rare the disease is, because the healthy majority produces many false alarms. The prior (how common the disease is) matters as much as the evidence (the test result).

## 2. Maths

$$P(H\mid D)=\frac{P(D\mid H)\,P(H)}{P(D)},\qquad P(D)=\sum_h P(D\mid h)P(h)$$

- $P(H)$ is the prior: belief before the data.
- $P(D\mid H)$ is the likelihood: how well the hypothesis explains the data.
- $P(H\mid D)$ is the posterior: belief after the data.
- $P(D)$ is a normaliser so the posteriors sum to 1.

Posterior is proportional to likelihood times prior.

## 3. The base-rate example

Prevalence 1%. Sensitivity (positive given sick) 90%. False positive rate (positive given healthy) 9%.

In [ ]:
prior = 0.01
sens = 0.90
fpr = 0.09

p_pos = sens * prior + fpr * (1 - prior)
posterior = sens * prior / p_pos
print("P(positive)         :", p_pos)
print("P(sick | positive)  :", posterior)

# Check by simulating 1,000,000 people
n = 1_000_000
sick = rng.random(n) < prior
positive = rng.random(n) < np.where(sick, sens, fpr)
print("simulated           :", sick[positive].mean())
print("sick among positives:", sick[positive].sum(), "of", positive.sum())

Out of about 98,000 positives, only 9,000 are sick. A positive result lifts the chance from 1% to 9%, not to 90%.

A second independent positive test uses the first posterior as the new prior.

In [ ]:
def update(prior: float, sens: float, fpr: float) -> float:
    return sens * prior / (sens * prior + fpr * (1 - prior))


p = prior
for i in range(1, 4):
    p = update(p, sens, fpr)
    print(f"after {i} positive test(s): {p:.3f}")

## 4. Beta-Binomial: updating a belief about a coin

Unknown coin bias $\theta$. Prior $\theta\sim\mathrm{Beta}(a,b)$. Observe $k$ heads in $n$ flips. The likelihood is $\theta^k(1-\theta)^{n-k}$ and the prior is proportional to $\theta^{a-1}(1-\theta)^{b-1}$. Multiply them:

$$P(\theta\mid k,n)\propto\theta^{a+k-1}(1-\theta)^{b+n-k-1}=\mathrm{Beta}(a+k,\,b+n-k)$$

The posterior is the same family as the prior, so updating is addition. This is called a conjugate prior. $a$ and $b$ act as pseudo-counts of heads and tails already seen. The posterior mean is $(a+k)/(a+b+n)$.

In [ ]:
def beta_update(a: float, b: float, k: int, n: int) -> tuple[float, float]:
    return a + k, b + n - k


a, b = 2, 2  # weak prior centred on 0.5
k, n = 7, 10  # 7 heads in 10 flips
a_post, b_post = beta_update(a, b, k, n)
print("posterior: Beta", (a_post, b_post), " mean:", a_post / (a_post + b_post))

# Check against a brute-force grid: prior * likelihood, normalised
theta = np.linspace(0.001, 0.999, 999)
unnorm = stats.beta.pdf(theta, a, b) * theta**k * (1 - theta) ** (n - k)
grid_post = unnorm / (unnorm.sum() * (theta[1] - theta[0]))
print("max gap to grid:", np.abs(grid_post - stats.beta.pdf(theta, a_post, b_post)).max())

plt.figure(figsize=(6, 3))
plt.plot(theta, stats.beta.pdf(theta, a, b), label="prior Beta(2, 2)")
plt.plot(theta, stats.beta.pdf(theta, a_post, b_post), label="posterior Beta(9, 5)")
plt.axvline(k / n, color="grey", ls=":", label="MLE 0.7")
plt.legend()
plt.show()

## 5. More data overwhelms the prior

Two people start with different priors about a coin whose true bias is 0.7: one sceptic who is sure it is fair, Beta(50, 50), and one with no opinion, Beta(1, 1). Feed them the same flips.

In [ ]:
true_theta = 0.7
flips = rng.random(1000) < true_theta
priors = {"sceptic Beta(50,50)": (50, 50), "flat Beta(1,1)": (1, 1)}

print(f"{'n':>5} " + " ".join(f"{name:>22}" for name in priors))
for n in (0, 10, 50, 200, 1000):
    k = int(flips[:n].sum())
    means = []
    for a0, b0 in priors.values():
        a1, b1 = beta_update(a0, b0, k, n)
        means.append(a1 / (a1 + b1))
    print(f"{n:>5} " + " ".join(f"{m:>22.3f}" for m in means))

The sceptic starts at 0.5 and is at 0.72 after 1000 flips, within 0.03 of the flat prior at 0.74 (the true value is 0.7, and 1000 flips of noise explain the rest). The prior is worth 100 pseudo-flips, so it fades once the real count is much bigger than that. At 10 flips it still dominates: 0.52 against 0.67.

## 6. Naive Bayes classifier

Classify by the posterior over classes $c$ given features $x_1,\dots,x_d$:

$$P(c\mid x)\propto P(c)\prod_{j=1}^d P(x_j\mid c)$$

"Naive" means the features are treated as independent given the class. That turns one hard joint density into $d$ easy one-dimensional ones. For continuous features use a Gaussian per class and feature, with mean and variance estimated from the training rows. Work in log space so the product does not underflow:

$$\log P(c\mid x)=\log P(c)+\sum_j\left[-\tfrac12\log(2\pi\sigma_{cj}^2)-\frac{(x_j-\mu_{cj})^2}{2\sigma_{cj}^2}\right]+\text{const}$$

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


class GaussianNaiveBayes:
    def __init__(self, var_smoothing: float = 1e-9) -> None:
        self.var_smoothing = var_smoothing

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GaussianNaiveBayes":
        self.classes_ = np.unique(y)
        self.log_prior_ = np.log(np.array([(y == c).mean() for c in self.classes_]))
        self.mean_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        self.var_ = np.array([X[y == c].var(axis=0) for c in self.classes_])
        # Same small floor sklearn uses, so a zero-variance feature does not divide by zero
        self.var_ += self.var_smoothing * X.var(axis=0).max()
        return self

    def log_joint(self, X: np.ndarray) -> np.ndarray:
        out = []
        for i in range(len(self.classes_)):
            ll = -0.5 * np.log(2 * np.pi * self.var_[i]) - (X - self.mean_[i]) ** 2 / (
                2 * self.var_[i]
            )
            out.append(self.log_prior_[i] + ll.sum(axis=1))
        return np.array(out).T

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        lj = self.log_joint(X)
        lj -= lj.max(axis=1, keepdims=True)  # stabilise before exponentiating
        p = np.exp(lj)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.classes_[self.log_joint(X).argmax(axis=1)]


mine = GaussianNaiveBayes().fit(X_train, y_train)
print("accuracy:", (mine.predict(X_test) == y_test).mean())

## 7. Library: sklearn GaussianNB

The two should agree on predictions and on probabilities.

In [ ]:
sk = GaussianNB().fit(X_train, y_train)
print("accuracy:", sk.score(X_test, y_test))
print("same predictions:", (sk.predict(X_test) == mine.predict(X_test)).all())
print("max probability gap:", np.abs(sk.predict_proba(X_test) - mine.predict_proba(X_test)).max())

## 8. What breaks

In [ ]:
# 1. Zero counts. Word counts for a spam class: the word "prize" never appeared in ham.
ham_count, ham_total, vocab = 0, 1000, 5000
print("P(prize | ham), raw      :", ham_count / ham_total)  # one zero wipes out the whole product
print("P(prize | ham), Laplace  :", (ham_count + 1) / (ham_total + vocab))

# 2. Correlated features are counted twice. Duplicate one iris column and watch the
# probabilities get more extreme even though no new information was added.
X_dup = np.hstack([X_train, X_train[:, [2]]])
dup = GaussianNaiveBayes().fit(X_dup, y_train)
row = int(mine.predict_proba(X_test).max(axis=1).argmin())  # the least certain test flower
print("single copy:", mine.predict_proba(X_test[[row]]))
print("duplicated :", dup.predict_proba(np.hstack([X_test, X_test[:, [2]]])[[row]]))

A zero count turns a strong piece of evidence into an absolute veto, so add one pseudo-count to every cell (Laplace smoothing). It is the same idea as the Beta(1,1) prior above. Repeated features shift the probabilities toward the extremes, so the class ranking often survives but the probabilities cannot be trusted as calibrated.

Next: where loss functions come from, and why a prior is the same thing as regularisation.